# 🧪 Notebook 03: n8n Webhook Interactive Tester & API Sandbox

This notebook provides an interactive testing environment to send characterization measurements directly to the **n8n Webhook Endpoint** (`http://localhost:5678/webhook-test/ingest-characterization`).

---

### 📌 Features:
1. **Interactive Function:** Test any characterization technique with custom metadata.
2. **Success Ingestion Test:** Verify clean HTTP 200 responses and PostgreSQL insertion.
3. **Fallback Error Test:** Verify HTTP 400 error responses and `audit_log` tracking for invalid inputs.
4. **All 9 Techniques Batch Tester:** Test all multimodal characterizations with 1 click.


In [ ]:
import os
import sys
import json
import requests

# Default Webhook URLs
TEST_WEBHOOK_URL = "http://localhost:5678/webhook-test/ingest-characterization"
PROD_WEBHOOK_URL = "http://localhost:5678/webhook/ingest-characterization"

def send_n8n_webhook(file_path, technique='PXRD', sample_code='RS1883', operator_name='Rubén S.N.', use_test_url=True):
    """
    Sends characterization file and metadata payload to n8n Webhook.
    """
    url = TEST_WEBHOOK_URL if use_test_url else PROD_WEBHOOK_URL
    
    # Resolve path if running from notebooks/ directory
    if not os.path.exists(file_path):
        parent_path = os.path.join('..', file_path)
        if os.path.exists(parent_path):
            file_path = parent_path
            
    if not os.path.exists(file_path):
        print(f"❌ Error: File not found at {file_path}")
        return
        
    print(f"🚀 Sending HTTP POST to n8n Webhook ({url})...")
    print(f" ├── Sample Code: {sample_code}")
    print(f" ├── Technique:   {technique}")
    print(f" ├── Operator:    {operator_name}")
    print(f" └── File Path:   {file_path} ({os.path.getsize(file_path)} bytes)\n")
    
    payload = {
        'sample_code': sample_code,
        'technique': technique,
        'operator_name': operator_name
    }
    
    try:
        with open(file_path, 'rb') as f:
            files = {'data_file': (os.path.basename(file_path), f)}
            response = requests.post(url, data=payload, files=files, timeout=15)
            
        print(f"📡 Response Status Code: {response.status_code}")
        try:
            res_json = response.json()
            print("📄 Returned JSON Payload:")
            print(json.dumps(res_json, indent=2, ensure_ascii=False))
        except Exception:
            print("📄 Returned Raw Response:")
            print(response.text)
            
        if response.status_code == 200:
            print("\n✅ SUCCESS: Measurement processed by n8n!")
        else:
            print(f"\n⚠️ ALERT: Webhook returned status {response.status_code}")
            
    except Exception as e:
        print(f"❌ Connection Error: {e}")

print("✅ Webhook Tester Function initialized!")


## 🟢 1. Test Single Characterization Ingestion (PXRD Sample)

In [ ]:
# Test sending a PXRD sample file
send_n8n_webhook(
    file_path=r'data/sample_datalake_demo/PXRD/24DRX0180-RS1883_CoPy.xrdml',
    technique='PXRD',
    sample_code='RS1883',
    operator_name='Rubén S.N.',
    use_test_url=True
)


## 🔴 2. Test Fallback Error Branch (Invalid Technique)

In [ ]:
# Test sending an invalid technique name to trigger the HTTP 400 Fallback error branch
send_n8n_webhook(
    file_path=r'data/sample_datalake_demo/PXRD/24DRX0180-RS1883_CoPy.xrdml',
    technique='INVALID_TECHNIQUE',
    sample_code='RS1883',
    operator_name='Rubén S.N.',
    use_test_url=True
)


## 🧪 3. Test All Multimodal Techniques Loop

In [ ]:
# Array of representative sample files across all techniques
samples = [
    ('PXRD', r'data/sample_datalake_demo/PXRD/24DRX0180-RS1883_CoPy.xrdml', 'RS1883'),
    ('VT-XRD', r'data/sample_datalake_demo/VT-XRD/2-DRXB-24001(1)-RSMix10_MgTriaz_0 30°C PB1mm.brml', 'RSMIX10'),
    ('TGA', r'data/sample_datalake_demo/TGA/IEF-22(Fe)_Ar_1000°C_RS1104.txt', 'RS1104'),
    ('FTIR', r'data/sample_datalake_demo/FTIR/RS1578_FePy.csv', 'RS1578'),
    ('BET', r'data/sample_datalake_demo/BET/24PT0126-RSMix15-CoPyF2-st1.XLS', 'RSMIX15'),
    ('UV_VIS', r'data/sample_datalake_demo/UV_Vis/2021-09-03 - IEF-8.Sample.asc', 'IEF-8'),
    ('SEM', r'data/sample_datalake_demo/SEM/CaPyDur_2d(x200).jpg', 'CAPYDUR'),
]

print(f"Testing {len(samples)} techniques sequentially...")
for tech, fpath, code in samples:
    print(f"\n------------------ [{tech}] ------------------")
    send_n8n_webhook(fpath, technique=tech, sample_code=code, use_test_url=True)
